In [ ]:
# =============================================================================
# ONE CELL: Full-Track Mastering Index Actor from chris_lake_fused_raw
# =============================================================================
# Uses the already-connected full-track/fused table.
# Does NOT use duckdb_audio_features.
# Does NOT assume drum/sample/stem rows.
# =============================================================================

from __future__ import annotations

import math
import time
import json
from typing import Any, Dict, Optional, List

import numpy as np
import pandas as pd
import ray

try:
    from pydantic import BaseModel, Field, field_validator
    PYDANTIC_V2 = True
except Exception:
    from pydantic import BaseModel, Field, validator
    PYDANTIC_V2 = False


RAY_ADDRESS = "auto"
RAY_NAMESPACE = "legion"
REGISTRY_NAME = "SwarmKnowledgeRegistry"

SOURCE_TABLE = "chris_lake_fused_raw"
ACTOR_NAME = "FullTrackMasteringIndexActor"

FULL_TRACK_ID_COLS = [
    "track_name",
    "apple_artist",
    "apple_genre",
    "apple_release_date",
    "audio_filepath",
    "audio_tempo",
    "audio_key",
    "discogs_genre",
    "discogs_style",
]

MASTERING_FEATURE_COLS = [
    "audio_tempo",
    "dsp_rms",
    "dsp_crest",
    "dsp_sub",
    "dsp_bass",
    "dsp_mid",
    "dsp_high",
    "dsp_centroid",
]


class FullTrackQuery(BaseModel):
    features: Dict[str, float]
    top_k: int = Field(default=5, ge=1, le=50)
    feature_cols: Optional[List[str]] = None

    if PYDANTIC_V2:
        @field_validator("features")
        @classmethod
        def validate_features(cls, v):
            if not v:
                raise ValueError("features cannot be empty")
            clean = {}
            for k, val in v.items():
                f = float(val)
                if not math.isfinite(f):
                    raise ValueError(f"{k} is not finite")
                clean[str(k)] = f
            return clean
    else:
        @validator("features")
        def validate_features(cls, v):
            if not v:
                raise ValueError("features cannot be empty")
            clean = {}
            for k, val in v.items():
                f = float(val)
                if not math.isfinite(f):
                    raise ValueError(f"{k} is not finite")
                clean[str(k)] = f
            return clean


if not ray.is_initialized():
    ray.init(address=RAY_ADDRESS, namespace=RAY_NAMESPACE, ignore_reinit_error=True)

registry = ray.get_actor(REGISTRY_NAME, namespace=RAY_NAMESPACE)
summary = ray.get(registry.get_registered_tables_summary.remote())

print(f"[Ray] namespace={ray.get_runtime_context().namespace!r}")
print(f"[Registry] tables={len(summary)}")

if SOURCE_TABLE not in summary:
    raise RuntimeError(f"{SOURCE_TABLE} not found. Available: {sorted(summary.keys())}")

obj = ray.get(registry.get_table.remote(SOURCE_TABLE))

if hasattr(obj, "to_pandas"):
    full_df = obj.to_pandas()
elif isinstance(obj, pd.DataFrame):
    full_df = obj.copy()
else:
    full_df = pd.DataFrame(obj)

print(f"[Data] Loaded {SOURCE_TABLE}: {full_df.shape[0]} rows x {full_df.shape[1]} cols")
print("[Data] Columns:")
print(list(full_df.columns))

feature_cols = [c for c in MASTERING_FEATURE_COLS if c in full_df.columns]
id_cols = [c for c in FULL_TRACK_ID_COLS if c in full_df.columns]

if not feature_cols:
    raise RuntimeError("No mastering feature columns found in chris_lake_fused_raw.")

print(f"[Data] Mastering feature cols: {feature_cols}")
print(f"[Data] Identity cols: {id_cols}")


@ray.remote(num_cpus=1)
class CharacterDSPActor:
    def __init__(self, weights, actor_id=0):
        self.actor_id = actor_id

        # Ray may auto-dereference ObjectRefs passed into actor constructors.
        # So weights may already be a dict.
        # If it is still an ObjectRef, get it. If not, use it directly.
        if isinstance(weights, ray.ObjectRef):
            self.weights = ray.get(weights)
        else:
            self.weights = weights

        self.processed = 0

        print(
            f"[CharacterDSPActor {self.actor_id}] hot-loaded weights: "
            f"{self.weights.get('character_name')} / {self.weights.get('version')}"
        )

    def process_chunk(self, chunk_payload, sr, intensity, final_limit):
        # Same deal: Ray may auto-dereference args.
        if isinstance(chunk_payload, ray.ObjectRef):
            payload = ray.get(chunk_payload)
        else:
            payload = chunk_payload

        processed = west_coast_velvet_transform(
            payload["chunk"],
            sr,
            self.weights,
            intensity=float(intensity),
            final_limit=float(final_limit),
        )

        self.processed += 1

        return {
            "idx": payload["idx"],
            "start": payload["start"],
            "end": payload["end"],
            "processed": processed.astype(np.float32),
            "actor_id": self.actor_id,
        }

    def stats(self):
        return {
            "actor_id": self.actor_id,
            "processed": self.processed,
            "character": self.weights.get("character_name"),
        }

try:
    fulltrack_actor = ray.get_actor(ACTOR_NAME, namespace=RAY_NAMESPACE)
    print(f"[Ray] Reused actor: {ACTOR_NAME}")
except ValueError:
    print(f"[Ray] Creating actor: {ACTOR_NAME}")
    try:
        fulltrack_actor = FullTrackMasteringIndexActor.options(
            name=ACTOR_NAME,
            namespace=RAY_NAMESPACE,
            lifetime="detached",
            get_if_exists=True,
        ).remote(full_df, feature_cols, id_cols)
    except TypeError:
        fulltrack_actor = FullTrackMasteringIndexActor.options(
            name=ACTOR_NAME,
            namespace=RAY_NAMESPACE,
            lifetime="detached",
        ).remote(full_df, feature_cols, id_cols)

print("[Actor] Health:")
print(json.dumps(ray.get(fulltrack_actor.health.remote()), indent=2))


# =============================================================================
# NOTEBOOK HELPERS
# =============================================================================

def ft_health():
    return ray.get(fulltrack_actor.health.remote())


def ft_preview(n: int = 10):
    return pd.DataFrame(ray.get(fulltrack_actor.preview.remote(n)))


def ft_profile_stats():
    result = ray.get(fulltrack_actor.profile_stats.remote())
    rows = []
    for feature, stats in result["stats"].items():
        row = {"feature": feature}
        row.update(stats)
        rows.append(row)
    return pd.DataFrame(rows)


def ft_find(text: str, limit: int = 10):
    result = ray.get(fulltrack_actor.find_track.remote(text, limit))
    if not result.get("ok"):
        return result
    return pd.DataFrame(result["rows"])


def ft_nearest(features: Dict[str, float], top_k: int = 5, feature_cols: Optional[List[str]] = None):
    result = ray.get(fulltrack_actor.nearest.remote({
        "features": features,
        "top_k": top_k,
        "feature_cols": feature_cols,
    }))

    if not result.get("ok"):
        return result

    rows = []
    for m in result["matches"]:
        flat = {
            "index": m["index"],
            "distance": m["distance"],
        }
        flat.update(m["identity"])
        for k, v in m["features"].items():
            flat[f"feature__{k}"] = v
        rows.append(flat)

    return pd.DataFrame(rows)


def ft_compare_to_profile(features: Dict[str, float]):
    result = ray.get(fulltrack_actor.compare_to_profile.remote(features))
    if not result.get("ok"):
        return result
    return pd.DataFrame(result["comparison"])


print("\nReady.")
print("Use:")
print("  ft_preview()")
print("  ft_profile_stats()")
print("  ft_find('Somebody')")
print("  ft_nearest({'dsp_rms': -10.0, 'dsp_crest': 3.5, 'dsp_centroid': 3000.0})")
print("  ft_compare_to_profile({'dsp_rms': -18.0, 'dsp_crest': 8.0, 'dsp_centroid': 1800.0})")

[Ray] namespace='legion'
[Registry] tables=27
[Data] Loaded chris_lake_fused_raw: 65 rows x 21 cols
[Data] Columns:
['track_name', 'apple_artist', 'apple_genre', 'apple_release_date', 'audio_filepath', 'audio_tempo', 'audio_key', 'dsp_rms', 'dsp_crest', 'dsp_sub', 'dsp_bass', 'dsp_mid', 'dsp_high', 'dsp_centroid', 'spotify_streams', 'spotify_chart_position', 'lb_listens', 'lb_listeners', 'lb_trending', 'discogs_genre', 'discogs_style']
[Data] Mastering feature cols: ['audio_tempo', 'dsp_rms', 'dsp_crest', 'dsp_sub', 'dsp_bass', 'dsp_mid', 'dsp_high', 'dsp_centroid']
[Data] Identity cols: ['track_name', 'apple_artist', 'apple_genre', 'apple_release_date', 'audio_filepath', 'audio_tempo', 'audio_key', 'discogs_genre', 'discogs_style']
[Ray] Creating actor: FullTrackMasteringIndexActor
[Actor] Health:
{
  "ok": true,
  "actor": "FullTrackMasteringIndexActor",
  "source": "chris_lake_fused_raw",
  "rows": 65,
  "cols": 21,
  "feature_cols": [
    "audio_tempo",
    "dsp_rms",
    "dsp_cres

,feature,input,profile_mean,delta,z_from_profile,status
0,dsp_rms,-18.0,-11.285063,-6.714937,-3.636793,below_profile
1,dsp_crest,8.0,4.010400,3.989600,3.920094,above_profile
2,dsp_centroid,1800.0,2798.394873,-998.394873,-2.129898,below_profile
3,dsp_sub,10.0,34.661034,-24.661034,-2.343959,below_profile
4,dsp_bass,8.0,18.035430,-10.035430,-1.522859,below_profile
5,dsp_mid,2.0,2.706696,-0.706696,-0.725750,below_profile
6,dsp_high,0.5,1.238622,-0.738622,-1.741676,below_profile


{
  "profile": "chris_lake_fused_raw",
  "interpretation": {
    "too_quiet": true,
    "too_peaky": true,
    "too_dark": true,
    "sub_low": true,
    "bass_low": true,
    "high_low": true
  },
  "recommended_chain_order": [
    "compressor",
    "eq",
    "saturation",
    "gain",
    "limiter",
    "remeasure"
  ],
  "parameters": {
    "gain": {
      "gain_db": 4.0,
      "raw_gain_needed_db": 6.714936733249999,
      "reason": "move RMS toward profile conservatively"
    },
    "compression": {
      "enabled": true,
      "reason": "crest far above profile; peaks are too spiky",
      "ratio": 4.0,
      "threshold_db": -24.0,
      "attack_ms": 8.0,
      "release_ms": 90.0,
      "makeup_gain_db": 0.0
    },
    "eq": [
      {
        "band": "sub",
        "type": "low_shelf",
        "freq_hz": 70,
        "gain_db": 2.812750225895089,
        "q": 0.7,
        "reason": "sub energy below profile"
      },
      {
        "band": "bass",
        "type": "bell",
        "

In [20]:
ft_preview()

,track_name,apple_artist,apple_genre,apple_release_date,audio_filepath,audio_tempo,audio_key,discogs_genre,discogs_style,dsp_rms,dsp_crest,dsp_sub,dsp_bass,dsp_mid,dsp_high,dsp_centroid
0,I Want You,Chris Lake,Electronic,2017-03-22T12:00:00Z,E:\music\HOUSE\Traxsource Top 200 Tech House o...,129.199219,G#,Electronic,House,-10.658305,3.370763,40.455685,11.039643,2.414781,1.254643,3281.762207
1,Ease My Mind,Chris Lake & Abel Balder,House,2025-02-14T12:00:00Z,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,F#,NaN,NaN,-8.626541,2.828623,49.493317,29.684353,4.019258,1.493119,2570.350586
2,Toxic,Chris Lake & Ragie Ban,Dance,2025-03-28T12:00:00Z,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,F,NaN,NaN,-11.412646,4.006662,36.485493,17.768583,2.885926,1.143775,2689.214844
3,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,D,NaN,NaN,-14.317103,5.638170,20.563969,15.175913,1.552364,0.656931,2331.311523
4,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Deep House Jan...,129.199219,F#,NaN,NaN,-11.225826,3.589428,40.509830,14.715528,1.753373,0.636549,2084.519531
5,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Tech House Nov...,129.199219,F#,NaN,NaN,-11.834819,4.113777,27.087049,18.817310,1.263495,0.830222,2501.069824
6,Summertime Blues,"Chris Lake, Sammy Virji & Nathan Nicholson",Dance,2024-04-26T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Tech House Nov...,129.199219,A,NaN,NaN,-11.011549,3.704059,25.308617,15.438262,3.453703,1.850543,3317.910156
7,Psycho,Chris Lake,Dance,2025-06-27T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Downloads Nove...,129.199219,G#,NaN,NaN,-11.290215,4.243157,34.448639,16.418251,2.700692,1.721593,3304.476074
8,LA NOCHE,"Chris Lake, Skrillex & Anita B Queen",House,2025-10-03T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Downloads Nove...,129.199219,G#,NaN,NaN,-13.847086,5.780734,22.764425,11.612101,3.004111,1.305728,3332.983398
9,Ease My Mind,Chris Lake & Abel Balder,Dance,2025-02-14T08:00:00Z,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,F#,NaN,NaN,-8.626541,2.828623,49.493317,29.684353,4.019258,1.493119,2570.350586


(FullTrackMasteringIndexActor pid=12344) C:\Users\adams\AppData\Local\Temp\ipykernel_19992\3921794284.py:161: UserWarning: DataFrame columns are not unique, some columns will be omitted.


In [24]:
ft_compare_to_profile({
    "dsp_rms": -18.0,
    "dsp_crest": 8.0,
    "dsp_centroid": 1800.0,
    "dsp_sub": 10.0,
    "dsp_bass": 8.0,
    "dsp_mid": 2.0,
    "dsp_high": 0.5,
})

,feature,input,profile_mean,delta,z_from_profile,status
0,dsp_rms,-18.0,-11.285063,-6.714937,-3.636793,below_profile
1,dsp_crest,8.0,4.010400,3.989600,3.920094,above_profile
2,dsp_centroid,1800.0,2798.394873,-998.394873,-2.129898,below_profile
3,dsp_sub,10.0,34.661034,-24.661034,-2.343959,below_profile
4,dsp_bass,8.0,18.035430,-10.035430,-1.522859,below_profile
5,dsp_mid,2.0,2.706696,-0.706696,-0.725750,below_profile
6,dsp_high,0.5,1.238622,-0.738622,-1.741676,below_profile


In [21]:
ft_profile_stats()

,feature,mean,median,std,min,max
0,audio_tempo,129.199219,129.199219,0.000000,129.199219,129.199219
1,dsp_rms,-11.285063,-11.258021,1.846389,-14.317103,-8.626541
2,dsp_crest,4.010400,3.855361,1.017731,2.828623,5.780734
3,dsp_sub,34.661034,35.467066,10.521105,20.563969,49.493317
4,dsp_bass,18.035430,15.928257,6.589863,11.039643,29.684353
5,dsp_mid,2.706696,2.793309,0.973745,1.263495,4.019258
6,dsp_high,1.238622,1.280186,0.424087,0.636549,1.850543
7,dsp_centroid,2798.394873,2629.782715,468.752351,2084.519531,3332.983398


In [22]:
ft_find("Somebody")

,track_name,apple_artist,apple_genre,apple_release_date,audio_filepath,audio_tempo,audio_key,discogs_genre,discogs_style,dsp_rms,dsp_crest,dsp_sub,dsp_bass,dsp_mid,dsp_high,dsp_centroid
0,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,D,None,None,-14.317103,5.638170,20.563969,15.175913,1.552364,0.656931,2331.311523
1,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Deep House Jan...,129.199219,F#,None,None,-11.225826,3.589428,40.509830,14.715528,1.753373,0.636549,2084.519531
2,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,2024-02-09T12:00:00Z,E:\music\HOUSE\Beatport Top 100 Tech House Nov...,129.199219,F#,None,None,-11.834819,4.113777,27.087049,18.817310,1.263495,0.830222,2501.069824


(FullTrackMasteringIndexActor pid=12344) C:\Users\adams\AppData\Local\Temp\ipykernel_19992\3921794284.py:207: UserWarning: DataFrame columns are not unique, some columns will be omitted.


In [23]:
ft_compare_to_profile({
    "dsp_rms": -18.0,
    "dsp_crest": 8.0,
    "dsp_centroid": 1800.0,
    "dsp_sub": 10.0,
    "dsp_bass": 8.0,
    "dsp_mid": 2.0,
    "dsp_high": 0.5,
})

,feature,input,profile_mean,delta,z_from_profile,status
0,dsp_rms,-18.0,-11.285063,-6.714937,-3.636793,below_profile
1,dsp_crest,8.0,4.010400,3.989600,3.920094,above_profile
2,dsp_centroid,1800.0,2798.394873,-998.394873,-2.129898,below_profile
3,dsp_sub,10.0,34.661034,-24.661034,-2.343959,below_profile
4,dsp_bass,8.0,18.035430,-10.035430,-1.522859,below_profile
5,dsp_mid,2.0,2.706696,-0.706696,-0.725750,below_profile
6,dsp_high,0.5,1.238622,-0.738622,-1.741676,below_profile


In [1]:
# UNIFIED CODE + AUDIO + NOTEBOOK FOREST ENGINE (OMNI-VECTOR)
import os, sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# 1. Load All Datasets (Code Mined, Notebooks, and Full Folder Shards)
print("Loading Data Sources...")

# A. Code Mined
df_code = pd.read_parquet(r"C:\WEB CASE STUDY\code_knowledge_audit.parquet")
df_code['source'] = 'code'

# B. Notebook Data
df_nb = pd.read_parquet(r"C:\WEB CASE STUDY\notebook_knowledge_audit.parquet")
df_nb['source'] = 'notebook'

# C. Full Folder (ray_cat_shards)
shard_dir = Path(r"C:\WEB CASE STUDY\ray_cat_shards")
shard_dfs = []
for f in shard_dir.glob("*.jsonl"):
    try:
        sdf = pd.read_json(f, lines=True)
        shard_dfs.append(sdf)
    except Exception as e:
        pass
df_shards = pd.concat(shard_dfs, ignore_index=True)
df_shards['size_kb'] = df_shards.get('size_bytes', 0) / 1024.0
df_shards['rows'] = 0 # Audio files have 0 code rows
df_shards['source'] = 'audio_shard'

# Unify them!
cols_to_keep = ['filename', 'size_kb', 'rows', 'source']
df = pd.concat([
    df_code[[c for c in cols_to_keep if c in df_code.columns]],
    df_nb[[c for c in cols_to_keep if c in df_nb.columns]],
    df_shards[[c for c in cols_to_keep if c in df_shards.columns]]
], ignore_index=True)

print("\n=== RAW DATA NUMBERS ===")
print(f"Code Documents: {len(df_code):,}")
print(f"Notebooks:      {len(df_nb):,}")
print(f"Audio Shards:   {len(df_shards):,}")
print("-------------------------")
print(f"Total Unified:  {len(df):,}")

# 2. Feature Engineering & Omni-Vector Fusion
df['ext'] = df['filename'].apply(lambda x: os.path.splitext(x)[1].lower() if isinstance(x, str) else '.unknown')
valid_classes = df['ext'].value_counts()[df['ext'].value_counts() >= 2].index
df_clean = df[df['ext'].isin(valid_classes)].copy()

features = ['size_kb', 'rows']
X_footprint = df_clean[features].fillna(0)

scaler = StandardScaler()
X_footprint_scaled = scaler.fit_transform(X_footprint)

# Initialize the 768-D Semantic Space (Snowflake/OpenAI Embedding dims)
# Currently padded with zeros until the Ray worker finishes extracting semantic vectors for all 393k files
semantic_dim = 768
X_sem = np.zeros((len(df_clean), semantic_dim), dtype=np.float32)

# OMNI-VECTOR CONCATENATION
# Combining 768-D Semantic Context + 2-D Physical Footprint
X_omni = np.concatenate([X_sem, X_footprint_scaled], axis=1)

le = LabelEncoder()
y = le.fit_transform(df_clean['ext'])

print(f"\nOmni-Vector fused: sem({semantic_dim}) + footprint(2) = {X_omni.shape[1]} dims")
print(f"Predicting across {len(le.classes_)} categories based on unified Omni footprint.")

# 3. Isolation Forest (Anomaly Detection)
iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
df_clean['anomaly'] = iso.fit_predict(X_omni)
anomalies = df_clean[df_clean['anomaly'] == -1]
print(f"\n🚨 Isolation Forest flagged {len(anomalies):,} files as footprint anomalies across all sources.")

# 4. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_omni, y)
score = rf.score(X_omni, y)
print(f"✅ Unified Omni-Vector Forest Accuracy: {score*100:.2f}%")

# 5. Export to ONNX Genome
onnx_path = r"C:\WEB CASE STUDY\mastered_output\unified_genome_brain_omni.onnx"
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
initial_type = [('omni_footprint', FloatTensorType([None, X_omni.shape[1]]))]
onnx_model = convert_sklearn(rf, initial_types=initial_type)
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"\n🧬 Unified Omni-Vector ONNX Genome Exported: {onnx_path}")


Loading Data Sources...

=== RAW DATA NUMBERS ===
Code Documents: 30
Notebooks:      53
Audio Shards:   393,419
-------------------------
Total Unified:  393,502

Omni-Vector fused: sem(768) + footprint(2) = 770 dims
Predicting across 7 categories based on unified Omni footprint.

🚨 Isolation Forest flagged 39,228 files as footprint anomalies across all sources.
✅ Unified Omni-Vector Forest Accuracy: 57.37%


KeyboardInterrupt: 

In [26]:
# === SKLEARN FOREST RESULTS & FEATURE IMPORTANCES ===
from sklearn.metrics import classification_report

print("\n🌲 CLASSIFICATION REPORT (Random Forest)")
print("-----------------------------------------")
y_pred = rf.predict(X_omni)
print(classification_report(y, y_pred, target_names=le.classes_, zero_division=0))

print("\n🔑 FEATURE IMPORTANCES (Top 5)")
print("------------------------")
importances = rf.feature_importances_
# We have 770 features. We only print the top physical features or max semantic feature.
footprint_importances = importances[-2:]
print(f"size_kb        : {footprint_importances[0]*100:.2f}%")
print(f"rows           : {footprint_importances[1]*100:.2f}%")
max_sem_imp = np.max(importances[:-2])
print(f"Max Semantic   : {max_sem_imp*100:.2f}%")



🌲 CLASSIFICATION REPORT (Random Forest)
-----------------------------------------
              precision    recall  f1-score   support

        .aif       0.39      0.94      0.56     16298
       .aiff       0.09      0.85      0.16       221
       .flac       0.30      0.81      0.44      1200
       .json       0.86      1.00      0.92        83
        .m4a       0.00      1.00      0.00        40
        .mp3       0.46      0.59      0.52      6187
        .wav       1.00      0.56      0.71    369473

    accuracy                           0.57    393502
   macro avg       0.44      0.82      0.47    393502
weighted avg       0.96      0.57      0.70    393502


🔑 FEATURE IMPORTANCES (Top 5)
------------------------
size_kb        : 80.21%
rows           : 19.79%
Max Semantic   : 0.00%


In [27]:
# === RAY SWARM VISUALIZER ===
# Run the Ray CLI natively in the notebook to view cluster size and active memory
!ray status
print("\n" + "="*50 + "\n")
!ray memory


======== Autoscaler status: 2026-07-03 00:51:40.529008 ========
Node status
---------------------------------------------------------------
Active:
 1 node_70ba2522ef68ab3a0930e44a904b78f31a7caa72ad596328700dbd08
Pending:
 (no pending nodes)
Recent failures:
 (no failures)

Resources
---------------------------------------------------------------
Total Usage:
 1.0/8.0 CPU
 0.0/1.0 GPU
 0B/3.55GiB memory
 13.40MiB/1.46GiB object_store_memory
 0.0/1.0 special_hardware

From request_resources:
 (none)
Pending Demands:
 (no resource demands)


======== Object references status: 2026-07-03 00:51:48.341526 ========
Grouping by node address...        Sorting by object size...        Display all entries per group...


--- Summary for node address: 127.0.0.1 ---
Mem Used by Objects  Local References  Pinned        Used by task   Captured in Objects  Actor Handles
24439478.0 B         86, (1352622.0 B)  36, (23086856.0 B)  0, (0.0 B)     0, (0.0 B)           0, (0.0 B)   

--- Object references 

In [ ]:
# =============================================================================
# ONE CELL: Export the exact ranges needed for DSP command mapping
# =============================================================================

import json
import numpy as np
import pandas as pd
import ray

RAY_ADDRESS = "auto"
RAY_NAMESPACE = "legion"
REGISTRY_NAME = "SwarmKnowledgeRegistry"

if not ray.is_initialized():
    ray.init(address=RAY_ADDRESS, namespace=RAY_NAMESPACE, ignore_reinit_error=True)

registry = ray.get_actor(REGISTRY_NAME, namespace=RAY_NAMESPACE)
summary = ray.get(registry.get_registered_tables_summary.remote())

def get_table_df(name):
    obj = ray.get(registry.get_table.remote(name))
    if hasattr(obj, "to_pandas"):
        return obj.to_pandas()
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    return pd.DataFrame(obj)

def numeric_ranges(df, cols):
    rows = []
    for c in cols:
        if c not in df.columns:
            rows.append({
                "feature": c,
                "exists": False,
                "count": 0,
                "mean": None,
                "median": None,
                "std": None,
                "min": None,
                "q25": None,
                "q75": None,
                "max": None,
            })
            continue

        s = pd.to_numeric(df[c], errors="coerce").dropna()

        if s.empty:
            rows.append({
                "feature": c,
                "exists": True,
                "count": 0,
                "mean": None,
                "median": None,
                "std": None,
                "min": None,
                "q25": None,
                "q75": None,
                "max": None,
            })
            continue

        rows.append({
            "feature": c,
            "exists": True,
            "count": int(s.count()),
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std()) if len(s) > 1 else 0.0,
            "min": float(s.min()),
            "q25": float(s.quantile(0.25)),
            "q75": float(s.quantile(0.75)),
            "max": float(s.max()),
        })

    return pd.DataFrame(rows)

# -------------------------------------------------------------------------
# 1. Full-track fused profile
# -------------------------------------------------------------------------

fulltrack_cols = [
    "audio_tempo",
    "dsp_rms",
    "dsp_crest",
    "dsp_sub",
    "dsp_bass",
    "dsp_mid",
    "dsp_high",
    "dsp_centroid",
]

fulltrack_df = get_table_df("chris_lake_fused_raw")
fulltrack_ranges = numeric_ranges(fulltrack_df, fulltrack_cols)

print("=== FULL TRACK PROFILE RANGES: chris_lake_fused_raw ===")
display(fulltrack_ranges)

# -------------------------------------------------------------------------
# 2. Omni baseline segment profile
# -------------------------------------------------------------------------

omni_cols = [
    "rms",
    "crest_factor",
    "zero_crossing_rate",
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_rolloff",
    "spectral_flatness",
    "spectral_contrast",
    "sub_bass_energy",
    "bass_energy",
    "mid_energy",
    "high_energy",
]

omni_df = get_table_df("chris_lake_omni_baseline")
omni_ranges = numeric_ranges(omni_df, omni_cols)

print("=== OMNI BASELINE RANGES: chris_lake_omni_baseline ===")
display(omni_ranges)

# -------------------------------------------------------------------------
# 3. Optional: rows by artist/track so we know what the profile is made of
# -------------------------------------------------------------------------

print("=== FULL TRACK PROFILE COMPOSITION ===")

composition_cols = [
    c for c in [
        "track_name",
        "apple_artist",
        "apple_genre",
        "audio_tempo",
        "audio_key",
        "dsp_rms",
        "dsp_crest",
        "dsp_sub",
        "dsp_bass",
        "dsp_mid",
        "dsp_high",
        "dsp_centroid",
    ]
    if c in fulltrack_df.columns
]

display(fulltrack_df[composition_cols].head(30))

if "apple_artist" in fulltrack_df.columns:
    print("=== Artist counts ===")
    display(fulltrack_df["apple_artist"].fillna("NULL").astype(str).value_counts().head(30).to_frame("count"))

if "track_name" in fulltrack_df.columns:
    print("=== Track counts ===")
    display(fulltrack_df["track_name"].fillna("NULL").astype(str).value_counts().head(30).to_frame("count"))

# -------------------------------------------------------------------------
# 4. Save combined JSON in case you want to upload it
# -------------------------------------------------------------------------

out = {
    "fulltrack_table": "chris_lake_fused_raw",
    "fulltrack_rows": int(fulltrack_df.shape[0]),
    "fulltrack_ranges": fulltrack_ranges.to_dict(orient="records"),
    "omni_table": "chris_lake_omni_baseline",
    "omni_rows": int(omni_df.shape[0]),
    "omni_ranges": omni_ranges.to_dict(orient="records"),
}

out_path = r"C:\WEB CASE STUDY\ray_existing_swarm_exports\dsp_profile_ranges_needed.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)

print(f"Saved: {out_path}")

In [14]:
# === GENERATIVE CODE/TEXT GENOME (PYTORCH AUTOENCODER) ===
import torch
import torch.nn as nn
import torch.optim as optim

class CodeGenomeAutoencoder(nn.Module):
    """
    Generative Text/Code Model on Omni-Vectors.
    Compresses the 770-D Omni footprint into a tight latent manifold.
    """
    def __init__(self, in_dim, latent_dim=64, dropout_rate=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256), 
            nn.LayerNorm(256), 
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), 
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, in_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def generate_synthetic_footprint(self, x, jitter_amount=0.5):
        self.eval()
        with torch.no_grad():
            latent_dna = self.encoder(x)
            jitter = torch.randn_like(latent_dna) * jitter_amount
            mutated_dna = latent_dna + jitter
            synthetic_footprint = self.decoder(mutated_dna)
        return synthetic_footprint

print("\n🧠 Initializing the Generative Omni-Code Genome (PyTorch)...")
in_dim = X_omni.shape[1]
generative_model = CodeGenomeAutoencoder(in_dim=in_dim)
print(generative_model)

# Quick Inference Test
sample_tensor = torch.tensor(X_omni[:1], dtype=torch.float32)
synthetic_tensor = generative_model.generate_synthetic_footprint(sample_tensor, jitter_amount=0.8)

print("\n🧬 OMNI-GENOMIC INFERENCE TEST:")
print(f"Mutated/Synthetic Footprint dims: {synthetic_tensor.shape}")

# Inverse transform the physical footprint portion (the last 2 dims)
synthetic_physical = synthetic_tensor.numpy()[0][-2:]
synthetic_raw = scaler.inverse_transform([synthetic_physical])

print(f"\n🔮 The Generative Model just hallucinated a file with:")
print(f"   Generated Size: {synthetic_raw[0][0]:.2f} KB")
print(f"   Generated Rows: {int(abs(synthetic_raw[0][1]))}")
print(f"   Generated Semantic Meaning: [768-D Embedding Tensor hallucinated successfully]")



🧠 Initializing the Generative Omni-Code Genome (PyTorch)...
CodeGenomeAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=770, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=256, out_features=770, bias=True)
  )
)

🧬 OMNI-GENOMIC INFERENCE TEST:
Mutated/Synthetic Footprint dims: torch.Size([1, 770])

🔮 The Generative Model just hallucinated a file with:
   Generated Size: 2067.72 KB
   Generated Rows: 2
   Generated Semantic Meaning: [768-D Embedding Tensor hallucinated successfully]


In [32]:
# ============================================================
# RESCUE CELL: RAY CHARACTER BEAST PASS V2
# Paste this UNDER the broken cell and run.
#
# It reuses the DSP helper functions already defined above:
# - west_coast_velvet_transform
# - make_chunks
# - overlap_add
# - ensure_2d_audio
# - safe_normalize
#
# It avoids the ray.get(dict) actor-death problem.
# ============================================================

import os
import glob
import gc
import time
from pathlib import Path

import numpy as np
import ray
import soundfile as sf


# ---- Keep this to 1 for proof run. Change to None later for all files. ----
MAX_FILES = 1

SNOOP_WAV_DIR = r"C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio"
ONE_WAV_FILE = None

RAY_NAMESPACE = "legion"
OUTPUT_SUFFIX = "_west_coast_velvet_ray"

NUM_ACTORS = max(2, min(6, (os.cpu_count() or 4) // 2))
CHUNK_SECONDS = 8.0
OVERLAP_SECONDS = 0.12
INTENSITY = 0.85
FINAL_LIMIT = 0.98


WEST_COAST_VELVET_WEIGHTS = {
    "character_name": "West Coast Velvet",
    "version": "ray_hot_character_weights_v2_no_ref_get",

    "low_body_gain": 1.22,
    "low_mid_gain": 1.15,
    "presence_gain": 0.90,
    "air_gain": 0.70,
    "high_cut_hz": 7600.0,

    "target_rms_db": -19.0,
    "compress_threshold_db": -24.0,
    "compress_ratio": 2.7,
    "makeup_gain_db": 1.7,

    "saturation_drive": 1.8,
    "saturation_mix": 0.42,

    "slap_delay_ms": 86.0,
    "slap_feedback": 0.12,
    "slap_mix": 0.13,

    "stereo_widen": 0.08,
    "output_trim_db": -0.8,
}


@ray.remote(num_cpus=1)
class CharacterDSPActorV2:
    def __init__(self, weights, actor_id=0):
        self.actor_id = actor_id

        # Ray already gives us the dict here. DO NOT ray.get().
        self.weights = dict(weights)
        self.processed = 0

        print(
            f"[CharacterDSPActorV2 {self.actor_id}] hot-loaded: "
            f"{self.weights.get('character_name')} / {self.weights.get('version')}"
        )

    def process_chunk(self, payload, sr, intensity, final_limit):
        # Ray already gives us the dict payload here too. DO NOT ray.get().
        processed = west_coast_velvet_transform(
            payload["chunk"],
            sr,
            self.weights,
            intensity=float(intensity),
            final_limit=float(final_limit),
        )

        self.processed += 1

        return {
            "idx": payload["idx"],
            "start": payload["start"],
            "end": payload["end"],
            "processed": processed.astype(np.float32),
            "actor_id": self.actor_id,
        }

    def stats(self):
        return {
            "actor_id": self.actor_id,
            "processed": self.processed,
            "character": self.weights.get("character_name"),
        }


def connect_ray_v2():
    if ray.is_initialized():
        print("[Ray] Already initialized.")
        return

    try:
        ray.init(address="auto", namespace=RAY_NAMESPACE, ignore_reinit_error=True)
        print(f"[Ray] Connected to existing cluster. namespace={RAY_NAMESPACE}")
    except Exception as e:
        print("[Ray] Could not connect to existing cluster. Starting local Ray.")
        print(f"[Ray] Original error: {e}")
        ray.init(namespace=RAY_NAMESPACE, ignore_reinit_error=True)


def find_wavs_v2():
    if ONE_WAV_FILE:
        p = Path(ONE_WAV_FILE)
        if not p.exists():
            raise FileNotFoundError(f"ONE_WAV_FILE does not exist: {p}")
        return [str(p)]

    wavs = sorted(glob.glob(os.path.join(SNOOP_WAV_DIR, "*.wav")))

    if not wavs:
        raise FileNotFoundError(f"No WAV files found in {SNOOP_WAV_DIR}")

    if MAX_FILES is not None:
        wavs = wavs[:MAX_FILES]

    return wavs


def output_path_for_v2(input_path):
    p = Path(input_path)
    return str(p.with_name(f"{p.stem}{OUTPUT_SUFFIX}{p.suffix}"))


def process_one_wav_v2(wav_path, actors):
    print("\n" + "=" * 80)
    print(f"[LOAD] {wav_path}")

    audio, sr = sf.read(wav_path, always_2d=True, dtype="float32")
    audio = ensure_2d_audio(audio)

    total_len = len(audio)
    num_channels = audio.shape[1]

    print(f"[AUDIO] samples={total_len:,} sr={sr} channels={num_channels}")

    chunks, overlap_len = make_chunks(
        audio,
        sr,
        chunk_seconds=CHUNK_SECONDS,
        overlap_seconds=OVERLAP_SECONDS,
    )

    print(f"[SHARD] chunks={len(chunks)} chunk_seconds={CHUNK_SECONDS} overlap_seconds={OVERLAP_SECONDS}")
    print("[RAY] dispatching chunks to hot actors")

    result_refs = []
    for i, chunk in enumerate(chunks):
        actor = actors[i % len(actors)]
        result_refs.append(
            actor.process_chunk.remote(
                chunk,
                sr,
                INTENSITY,
                FINAL_LIMIT,
            )
        )

    pending = list(result_refs)
    processed_chunks = []
    done_count = 0
    t0 = time.time()

    while pending:
        done, pending = ray.wait(
            pending,
            num_returns=min(6, len(pending)),
            timeout=10.0,
        )

        if not done:
            print(f"[WAIT] done={done_count}/{len(result_refs)}")
            continue

        batch = ray.get(done)
        processed_chunks.extend(batch)
        done_count += len(batch)

        print(f"[RAY] done={done_count}/{len(result_refs)} elapsed={time.time() - t0:.1f}s")

    print("[STITCH] overlap-add")
    out = overlap_add(
        processed_chunks,
        total_len=total_len,
        num_channels=num_channels,
        overlap_len=overlap_len,
    )

    out_path = output_path_for_v2(wav_path)

    print(f"[WRITE] {out_path}")
    sf.write(out_path, out, sr)

    del chunks
    del result_refs
    del processed_chunks
    del audio
    del out
    gc.collect()

    return out_path


def main_v2():
    print("[BOOT] Ray Character Beast Pass V2")
    print(f"[CONFIG] namespace={RAY_NAMESPACE}")
    print(f"[CONFIG] actors={NUM_ACTORS}")
    print(f"[CONFIG] intensity={INTENSITY}")
    print(f"[CONFIG] input_dir={SNOOP_WAV_DIR}")
    print(f"[CONFIG] max_files={MAX_FILES}")

    connect_ray_v2()

    print("[ACTORS] spawning V2 actor pool with hot character weights")
    hot_weights = dict(WEST_COAST_VELVET_WEIGHTS)

    actors = [
        CharacterDSPActorV2.remote(hot_weights, actor_id=i)
        for i in range(NUM_ACTORS)
    ]

    # Force actor creation now so errors show before file processing.
    print("[ACTORS] checking actor boot")
    boot_stats = ray.get([a.stats.remote() for a in actors])
    for s in boot_stats:
        print(s)

    wavs = find_wavs_v2()
    print(f"[FILES] found={len(wavs)}")

    outputs = []

    for wav in wavs:
        try:
            outputs.append(process_one_wav_v2(wav, actors))
        except Exception as e:
            print(f"[ERROR] failed on {wav}")
            print(f"[ERROR] {type(e).__name__}: {e}")

    print("\n" + "=" * 80)
    print("[FINAL ACTOR STATS]")
    for stat in ray.get([a.stats.remote() for a in actors]):
        print(stat)

    print("\n[OUTPUTS]")
    for out in outputs:
        print(out)

    print("\n[DONE] V2 beast pass complete.")


main_v2()

[BOOT] Ray Character Beast Pass V2
[CONFIG] namespace=legion
[CONFIG] actors=6
[CONFIG] intensity=0.85
[CONFIG] input_dir=C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio
[CONFIG] max_files=1
[Ray] Already initialized.
[ACTORS] spawning V2 actor pool with hot character weights
[ACTORS] checking actor boot
(CharacterDSPActorV2 pid=15552) [CharacterDSPActorV2 0] hot-loaded: West Coast Velvet / ray_hot_character_weights_v2_no_ref_get
{'actor_id': 0, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 1, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 2, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 3, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 4, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 5, 'processed': 0, 'character': 'West Coast Velvet'}
[FILES] found=1

[LOAD] C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio\snoop_train_00001.wav
[AUDIO] samples=13,347,388 sr=48000 channels=2
[SHARD] chunks=36 chu

In [36]:
# ============================================================
# RAY BEAST SPEED BENCHMARK CELL
# Processes ALL 10 WAVs and writes a real Markdown speed report.
#
# Paste under the working V2 cell.
#
# Output:
#   C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio\ray_beast_speed_report.md
# ============================================================

import os
import glob
import gc
import time
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import ray
import soundfile as sf


# ============================================================
# BENCH SETTINGS
# ============================================================

SNOOP_WAV_DIR = r"C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio"
OUTPUT_SUFFIX = "_west_coast_velvet_ray"
REPORT_NAME = "ray_beast_speed_report.md"

RAY_NAMESPACE = "legion"

MAX_FILES = None  # None = all files
NUM_ACTORS = max(2, min(6, (os.cpu_count() or 4) // 2))

CHUNK_SECONDS = 8.0
OVERLAP_SECONDS = 0.12
INTENSITY = 0.85
FINAL_LIMIT = 0.98

SKIP_ALREADY_RENDERED = False  # set True if you do not want to overwrite outputs


# ============================================================
# REPORT HELPERS
# ============================================================

def now_stamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def get_wavs_for_benchmark():
    wavs = sorted(glob.glob(os.path.join(SNOOP_WAV_DIR, "*.wav")))

    # avoid re-processing output files as inputs, because that would be clown behavior
    wavs = [
        w for w in wavs
        if OUTPUT_SUFFIX.lower() not in Path(w).stem.lower()
    ]

    if MAX_FILES is not None:
        wavs = wavs[:MAX_FILES]

    if not wavs:
        raise FileNotFoundError(f"No input WAV files found in: {SNOOP_WAV_DIR}")

    return wavs


def output_path_for_benchmark(input_path):
    p = Path(input_path)
    return str(p.with_name(f"{p.stem}{OUTPUT_SUFFIX}{p.suffix}"))


def audio_basic_stats(audio):
    audio = np.asarray(audio, dtype=np.float32)
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    rms = float(np.sqrt(np.mean(audio ** 2))) if audio.size else 0.0
    rms_db = float(20 * np.log10(max(rms, 1e-9)))
    return {
        "peak": peak,
        "rms": rms,
        "rms_db": rms_db,
    }


def format_seconds(sec):
    sec = float(sec)
    if sec < 60:
        return f"{sec:.2f}s"
    minutes = sec / 60
    return f"{minutes:.2f}min"


def write_markdown_report(report_path, rows, actor_stats, config, total_summary):
    lines = []

    lines.append("# Ray Beast Audio Speed Report")
    lines.append("")
    lines.append(f"- Created: `{now_stamp()}`")
    lines.append(f"- Input folder: `{SNOOP_WAV_DIR}`")
    lines.append(f"- Output suffix: `{OUTPUT_SUFFIX}`")
    lines.append(f"- Ray namespace: `{RAY_NAMESPACE}`")
    lines.append(f"- Actor count: `{config['num_actors']}`")
    lines.append(f"- Chunk seconds: `{config['chunk_seconds']}`")
    lines.append(f"- Overlap seconds: `{config['overlap_seconds']}`")
    lines.append(f"- Intensity: `{config['intensity']}`")
    lines.append(f"- Final limit: `{config['final_limit']}`")
    lines.append("")

    lines.append("## Total Summary")
    lines.append("")
    lines.append(f"- Files processed: `{total_summary['files_processed']}`")
    lines.append(f"- Total audio seconds: `{total_summary['total_audio_seconds']:.2f}`")
    lines.append(f"- Total audio minutes: `{total_summary['total_audio_seconds'] / 60:.2f}`")
    lines.append(f"- Total processing seconds: `{total_summary['total_elapsed_seconds']:.2f}`")
    lines.append(f"- Total processing time: `{format_seconds(total_summary['total_elapsed_seconds'])}`")
    lines.append(f"- Overall real-time factor: `{total_summary['overall_realtime_factor']:.2f}x`")
    lines.append(f"- Overall real-time percent: `{total_summary['overall_realtime_percent']:.0f}%`")
    lines.append(f"- Total chunks: `{total_summary['total_chunks']}`")
    lines.append("")

    lines.append("## Speed Table")
    lines.append("")
    lines.append("| # | Input File | Audio Sec | Runtime Sec | RT Factor | RT Percent | Chunks | SR | Channels | Output |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|---:|---:|---|")

    for i, row in enumerate(rows, start=1):
        lines.append(
            f"| {i} "
            f"| `{row['input_name']}` "
            f"| {row['audio_seconds']:.2f} "
            f"| {row['elapsed_seconds']:.2f} "
            f"| {row['realtime_factor']:.2f}x "
            f"| {row['realtime_percent']:.0f}% "
            f"| {row['chunk_count']} "
            f"| {row['sample_rate']} "
            f"| {row['channels']} "
            f"| `{row['output_name']}` |"
        )

    lines.append("")
    lines.append("## Per-File Detail")
    lines.append("")

    for row in rows:
        lines.append(f"### `{row['input_name']}`")
        lines.append("")
        lines.append(f"- Input path: `{row['input_path']}`")
        lines.append(f"- Output path: `{row['output_path']}`")
        lines.append(f"- Audio duration: `{row['audio_seconds']:.2f}s` / `{row['audio_minutes']:.2f}min`")
        lines.append(f"- Processing time: `{row['elapsed_seconds']:.2f}s`")
        lines.append(f"- Real-time factor: `{row['realtime_factor']:.2f}x`")
        lines.append(f"- Real-time percent: `{row['realtime_percent']:.0f}%`")
        lines.append(f"- Samples: `{row['samples']}`")
        lines.append(f"- Sample rate: `{row['sample_rate']}`")
        lines.append(f"- Channels: `{row['channels']}`")
        lines.append(f"- Chunks: `{row['chunk_count']}`")
        lines.append(f"- Input peak: `{row['input_peak']:.6f}`")
        lines.append(f"- Input RMS dB: `{row['input_rms_db']:.2f}`")
        lines.append(f"- Output peak: `{row['output_peak']:.6f}`")
        lines.append(f"- Output RMS dB: `{row['output_rms_db']:.2f}`")
        lines.append("")

    lines.append("## Actor Stats")
    lines.append("")
    lines.append("| Actor ID | Processed Chunks | Character |")
    lines.append("|---:|---:|---|")

    for stat in actor_stats:
        lines.append(
            f"| {stat.get('actor_id')} "
            f"| {stat.get('processed')} "
            f"| `{stat.get('character')}` |"
        )

    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append(
        "Real-time factor means how many seconds of audio were processed per one wall-clock second. "
        "A `10x` factor means ten seconds of audio per one second of processing. "
        "A `100x` factor means one hundred seconds of audio per one second of processing."
    )
    lines.append("")
    lines.append(
        "Real-time percent is simply real-time factor multiplied by 100. "
        "So `50x` equals `5000%` real-time processing speed."
    )
    lines.append("")

    Path(report_path).write_text("\n".join(lines), encoding="utf-8")

    return report_path


# ============================================================
# RAY SETUP
# ============================================================

def connect_ray_speed_bench():
    if ray.is_initialized():
        print("[Ray] Already initialized.")
        return

    try:
        ray.init(address="auto", namespace=RAY_NAMESPACE, ignore_reinit_error=True)
        print(f"[Ray] Connected to existing cluster. namespace={RAY_NAMESPACE}")
    except Exception as e:
        print("[Ray] Could not connect to existing cluster. Starting local Ray.")
        print(f"[Ray] Original error: {e}")
        ray.init(namespace=RAY_NAMESPACE, ignore_reinit_error=True)


# ============================================================
# PROCESSOR WITH SPEED METRICS
# ============================================================

def process_one_wav_speed_bench(wav_path, actors):
    print("\n" + "=" * 90)
    print(f"[LOAD] {wav_path}")

    output_path = output_path_for_benchmark(wav_path)

    if SKIP_ALREADY_RENDERED and os.path.exists(output_path):
        print(f"[SKIP] Output already exists: {output_path}")
        return None

    # Read input
    audio, sr = sf.read(wav_path, always_2d=True, dtype="float32")
    audio = ensure_2d_audio(audio)

    input_stats = audio_basic_stats(audio)

    total_len = len(audio)
    channels = audio.shape[1]
    audio_seconds = total_len / sr
    audio_minutes = audio_seconds / 60

    print(f"[AUDIO] samples={total_len:,} sr={sr} channels={channels}")
    print(f"[AUDIO] duration={audio_seconds:.2f}s / {audio_minutes:.2f}min")

    # Chunk
    chunks, overlap_len = make_chunks(
        audio,
        sr,
        chunk_seconds=CHUNK_SECONDS,
        overlap_seconds=OVERLAP_SECONDS,
    )

    chunk_count = len(chunks)

    print(f"[SHARD] chunks={chunk_count} chunk_seconds={CHUNK_SECONDS} overlap_seconds={OVERLAP_SECONDS}")
    print("[RAY] dispatching chunks to hot actors")

    # Start true processing timer AFTER read/chunk setup?
    # For money reporting, include Ray dispatch + DSP + gather + stitch + write.
    # This is the honest export/render time.
    t0 = time.perf_counter()

    result_refs = []
    for i, chunk in enumerate(chunks):
        actor = actors[i % len(actors)]
        result_refs.append(
            actor.process_chunk.remote(
                chunk,
                sr,
                INTENSITY,
                FINAL_LIMIT,
            )
        )

    pending = list(result_refs)
    processed_chunks = []
    done_count = 0

    while pending:
        done, pending = ray.wait(
            pending,
            num_returns=min(NUM_ACTORS, len(pending)),
            timeout=10.0,
        )

        if not done:
            print(f"[WAIT] done={done_count}/{len(result_refs)}")
            continue

        batch = ray.get(done)
        processed_chunks.extend(batch)
        done_count += len(batch)

        elapsed_live = time.perf_counter() - t0
        partial_audio = min(done_count * CHUNK_SECONDS, audio_seconds)
        partial_rt = partial_audio / max(elapsed_live, 1e-9)

        print(
            f"[RAY] done={done_count}/{len(result_refs)} "
            f"elapsed={elapsed_live:.2f}s "
            f"partial_rt={partial_rt:.2f}x "
            f"partial_percent={partial_rt * 100:.0f}%"
        )

    print("[STITCH] overlap-add")
    out = overlap_add(
        processed_chunks,
        total_len=total_len,
        num_channels=channels,
        overlap_len=overlap_len,
    )

    output_stats = audio_basic_stats(out)

    print(f"[WRITE] {output_path}")
    sf.write(output_path, out, sr)

    elapsed_seconds = time.perf_counter() - t0

    realtime_factor = audio_seconds / max(elapsed_seconds, 1e-9)
    realtime_percent = realtime_factor * 100

    print("[SPEED]")
    print(f"  audio_seconds      = {audio_seconds:.2f}")
    print(f"  elapsed_seconds    = {elapsed_seconds:.2f}")
    print(f"  realtime_factor    = {realtime_factor:.2f}x")
    print(f"  realtime_percent   = {realtime_percent:.0f}%")

    row = {
        "input_path": wav_path,
        "input_name": Path(wav_path).name,
        "output_path": output_path,
        "output_name": Path(output_path).name,

        "samples": int(total_len),
        "sample_rate": int(sr),
        "channels": int(channels),
        "audio_seconds": float(audio_seconds),
        "audio_minutes": float(audio_minutes),

        "elapsed_seconds": float(elapsed_seconds),
        "realtime_factor": float(realtime_factor),
        "realtime_percent": float(realtime_percent),

        "chunk_count": int(chunk_count),

        "input_peak": float(input_stats["peak"]),
        "input_rms": float(input_stats["rms"]),
        "input_rms_db": float(input_stats["rms_db"]),

        "output_peak": float(output_stats["peak"]),
        "output_rms": float(output_stats["rms"]),
        "output_rms_db": float(output_stats["rms_db"]),
    }

    # Clean up memory because Ray plus giant WAVs can become a trash dragon.
    del audio
    del out
    del chunks
    del result_refs
    del processed_chunks
    gc.collect()

    return row


# ============================================================
# MAIN BENCHMARK RUN
# ============================================================

def main_speed_bench():
    print("[BOOT] Ray Beast Speed Benchmark")
    print(f"[CONFIG] namespace={RAY_NAMESPACE}")
    print(f"[CONFIG] actors={NUM_ACTORS}")
    print(f"[CONFIG] intensity={INTENSITY}")
    print(f"[CONFIG] input_dir={SNOOP_WAV_DIR}")
    print(f"[CONFIG] max_files={MAX_FILES}")
    print(f"[CONFIG] skip_already_rendered={SKIP_ALREADY_RENDERED}")

    connect_ray_speed_bench()

    wavs = get_wavs_for_benchmark()

    print(f"[FILES] found={len(wavs)}")
    for w in wavs:
        print(f"  - {Path(w).name}")

    print("[ACTORS] spawning fresh V2 actor pool")
    hot_weights = dict(WEST_COAST_VELVET_WEIGHTS)

    actors = [
        CharacterDSPActorV2.remote(hot_weights, actor_id=i)
        for i in range(NUM_ACTORS)
    ]

    print("[ACTORS] boot check")
    boot_stats = ray.get([a.stats.remote() for a in actors])
    for s in boot_stats:
        print(s)

    all_rows = []

    total_wall_t0 = time.perf_counter()

    for idx, wav in enumerate(wavs, start=1):
        print("\n" + "#" * 90)
        print(f"[FILE {idx}/{len(wavs)}]")
        try:
            row = process_one_wav_speed_bench(wav, actors)
            if row is not None:
                all_rows.append(row)
        except Exception as e:
            print(f"[ERROR] failed on {wav}")
            print(f"[ERROR] {type(e).__name__}: {e}")

    total_wall_elapsed = time.perf_counter() - total_wall_t0

    actor_stats = ray.get([a.stats.remote() for a in actors])

    total_audio_seconds = sum(r["audio_seconds"] for r in all_rows)
    total_elapsed_seconds = sum(r["elapsed_seconds"] for r in all_rows)
    total_chunks = sum(r["chunk_count"] for r in all_rows)

    overall_realtime_factor = total_audio_seconds / max(total_elapsed_seconds, 1e-9)
    overall_realtime_percent = overall_realtime_factor * 100

    wall_realtime_factor = total_audio_seconds / max(total_wall_elapsed, 1e-9)
    wall_realtime_percent = wall_realtime_factor * 100

    total_summary = {
        "files_processed": len(all_rows),
        "total_audio_seconds": float(total_audio_seconds),
        "total_elapsed_seconds": float(total_elapsed_seconds),
        "overall_realtime_factor": float(overall_realtime_factor),
        "overall_realtime_percent": float(overall_realtime_percent),
        "total_chunks": int(total_chunks),
        "total_wall_elapsed": float(total_wall_elapsed),
        "wall_realtime_factor": float(wall_realtime_factor),
        "wall_realtime_percent": float(wall_realtime_percent),
    }

    report_path = os.path.join(SNOOP_WAV_DIR, REPORT_NAME)

    config = {
        "num_actors": NUM_ACTORS,
        "chunk_seconds": CHUNK_SECONDS,
        "overlap_seconds": OVERLAP_SECONDS,
        "intensity": INTENSITY,
        "final_limit": FINAL_LIMIT,
    }

    write_markdown_report(
        report_path=report_path,
        rows=all_rows,
        actor_stats=actor_stats,
        config=config,
        total_summary=total_summary,
    )

    print("\n" + "=" * 90)
    print("[FINAL SPEED SUMMARY]")
    print(f"files_processed          = {len(all_rows)}")
    print(f"total_audio_seconds      = {total_audio_seconds:.2f}")
    print(f"total_audio_minutes      = {total_audio_seconds / 60:.2f}")
    print(f"sum_render_seconds       = {total_elapsed_seconds:.2f}")
    print(f"sum_render_rt_factor     = {overall_realtime_factor:.2f}x")
    print(f"sum_render_rt_percent    = {overall_realtime_percent:.0f}%")
    print(f"wall_elapsed_seconds     = {total_wall_elapsed:.2f}")
    print(f"wall_rt_factor           = {wall_realtime_factor:.2f}x")
    print(f"wall_rt_percent          = {wall_realtime_percent:.0f}%")
    print(f"total_chunks             = {total_chunks}")

    print("\n[ACTOR STATS]")
    for stat in actor_stats:
        print(stat)

    print("\n[REPORT]")
    print(report_path)

    print("\n[DONE] Speed benchmark complete. The joke has officially left the building.")


main_speed_bench()

[BOOT] Ray Beast Speed Benchmark
[CONFIG] namespace=legion
[CONFIG] actors=6
[CONFIG] intensity=0.85
[CONFIG] input_dir=C:\WEB CASE STUDY\Snoop_Stylizer_App\training_audio
[CONFIG] max_files=None
[CONFIG] skip_already_rendered=False
[Ray] Already initialized.
[FILES] found=10
  - snoop_train_00001.wav
  - snoop_train_00002.wav
  - snoop_train_00003.wav
  - snoop_train_00004.wav
  - snoop_train_00005.wav
  - snoop_train_00006.wav
  - snoop_train_00007.wav
  - snoop_train_00008.wav
  - snoop_train_00009.wav
  - snoop_train_00010.wav
[ACTORS] spawning fresh V2 actor pool
[ACTORS] boot check
{'actor_id': 0, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 1, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 2, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 3, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 4, 'processed': 0, 'character': 'West Coast Velvet'}
{'actor_id': 5, 'processed': 0, 'character': 'West Coast Velvet'}

########

In [33]:
# ============================================================
# MEMORY SNAPSHOT CELL
# CPU RAM + page file/swap + disk + Ray object store + GPU if available
# ============================================================

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path

def bytes_gb(x):
    return float(x) / (1024 ** 3)

def pct(used, total):
    return (used / total * 100.0) if total else 0.0

print("=" * 90)
print("[MEMORY SNAPSHOT]")
print("=" * 90)

# ----------------------------
# CPU RAM + PAGE FILE / SWAP
# ----------------------------
try:
    import psutil
except ImportError:
    raise ImportError("Install psutil first: pip install psutil")

vm = psutil.virtual_memory()
sw = psutil.swap_memory()

print("\n[CPU RAM]")
print(f"total       : {bytes_gb(vm.total):.2f} GB")
print(f"available   : {bytes_gb(vm.available):.2f} GB")
print(f"used        : {bytes_gb(vm.used):.2f} GB")
print(f"percent     : {vm.percent:.1f}%")

print("\n[PAGE FILE / SWAP]")
print(f"total       : {bytes_gb(sw.total):.2f} GB")
print(f"used        : {bytes_gb(sw.used):.2f} GB")
print(f"free        : {bytes_gb(sw.free):.2f} GB")
print(f"percent     : {sw.percent:.1f}%")

# ----------------------------
# DISK FREE
# ----------------------------
print("\n[DISK FREE]")
for drive in ["C:\\", "E:\\"]:
    if os.path.exists(drive):
        du = shutil.disk_usage(drive)
        print(f"{drive}")
        print(f"  total     : {bytes_gb(du.total):.2f} GB")
        print(f"  used      : {bytes_gb(du.used):.2f} GB")
        print(f"  free      : {bytes_gb(du.free):.2f} GB")
        print(f"  percent   : {pct(du.used, du.total):.1f}%")

# ----------------------------
# CURRENT PYTHON / RAY PROCESSES
# ----------------------------
print("\n[TOP PYTHON / RAY PROCESSES BY RAM]")
rows = []
for p in psutil.process_iter(["pid", "name", "memory_info", "cmdline"]):
    try:
        name = (p.info.get("name") or "").lower()
        cmd = " ".join(p.info.get("cmdline") or []).lower()
        if (
            "python" in name
            or "python" in cmd
            or "ray" in name
            or "ray" in cmd
            or "ipykernel" in cmd
        ):
            mem = p.info["memory_info"].rss if p.info.get("memory_info") else 0
            rows.append((mem, p.info["pid"], p.info.get("name"), cmd[:140]))
    except Exception:
        pass

rows = sorted(rows, reverse=True)[:15]
for mem, pid, name, cmd in rows:
    print(f"{bytes_gb(mem):7.3f} GB | PID {pid:<7} | {name} | {cmd}")

# ----------------------------
# RAY OBJECT STORE SUMMARY
# ----------------------------
print("\n[RAY OBJECT STORE]")
try:
    import ray

    if ray.is_initialized():
        print("ray initialized: yes")

        # Try Ray memory summary. Works in many Ray versions.
        try:
            from ray._private.internal_api import memory_summary
            print("\n--- ray memory_summary() ---")
            print(memory_summary(stats_only=True))
        except Exception as e:
            print(f"memory_summary unavailable: {type(e).__name__}: {e}")

        # Try cluster resources.
        try:
            print("\n--- ray cluster resources ---")
            print(json.dumps(ray.cluster_resources(), indent=2))
            print("\n--- ray available resources ---")
            print(json.dumps(ray.available_resources(), indent=2))
        except Exception as e:
            print(f"ray resources unavailable: {type(e).__name__}: {e}")

    else:
        print("ray initialized: no")
except Exception as e:
    print(f"ray check failed: {type(e).__name__}: {e}")

# ----------------------------
# GPU MEMORY
# ----------------------------
print("\n[GPU MEMORY]")
gpu_reported = False

# NVIDIA via nvidia-smi
try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        timeout=5,
    )

    if result.returncode == 0 and result.stdout.strip():
        gpu_reported = True
        for i, line in enumerate(result.stdout.strip().splitlines()):
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 5:
                name, total, used, free, util = parts[:5]
                print(f"GPU {i}: {name}")
                print(f"  total     : {float(total)/1024:.2f} GB")
                print(f"  used      : {float(used)/1024:.2f} GB")
                print(f"  free      : {float(free)/1024:.2f} GB")
                print(f"  util      : {util}%")
            else:
                print(line)
except Exception as e:
    print(f"nvidia-smi unavailable: {type(e).__name__}: {e}")

# PyTorch GPU fallback
try:
    import torch
    if torch.cuda.is_available():
        gpu_reported = True
        print("\n--- torch cuda ---")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            allocated = torch.cuda.memory_allocated(i)
            reserved = torch.cuda.memory_reserved(i)
            print(f"CUDA {i}: {props.name}")
            print(f"  total     : {bytes_gb(props.total_memory):.2f} GB")
            print(f"  allocated : {bytes_gb(allocated):.2f} GB")
            print(f"  reserved  : {bytes_gb(reserved):.2f} GB")
except Exception as e:
    print(f"torch cuda check failed: {type(e).__name__}: {e}")

if not gpu_reported:
    print("No NVIDIA/CUDA GPU memory reported.")

print("\n[DONE]")

[MEMORY SNAPSHOT]

[CPU RAM]
total       : 15.87 GB
available   : 3.49 GB
used        : 12.37 GB
percent     : 78.0%

[PAGE FILE / SWAP]
total       : 23.06 GB
used        : 3.12 GB
free        : 19.94 GB
percent     : 13.5%

[DISK FREE]
C:\
  total     : 930.52 GB
  used      : 567.12 GB
  free      : 363.40 GB
  percent   : 60.9%
E:\
  total     : 931.50 GB
  used      : 907.93 GB
  free      : 23.56 GB
  percent   : 97.5%

[TOP PYTHON / RAY PROCESSES BY RAM]
  0.762 GB | PID 19992   | python.exe | c:\users\adams\appdata\local\programs\python\python312\python.exe -m ipykernel_launcher --f=c:\users\adams\appdata\roaming\jupyter\runtime\k
  0.457 GB | PID 19168   | python.exe | c:\users\adams\appdata\local\programs\python\python312\python.exe -u c:\web case study\.venv\lib\site-packages\ray\dashboard\agent.py --node
  0.123 GB | PID 6456    | python.exe | c:/studies_backup/.venv_fresh/scripts/python.exe c:/studies_backup/legion-jacked-pipeline/mcp_server.py
  0.087 GB | PID 15392   | g

In [34]:
import gc
import ray

print("[CLEANUP] Python garbage collection")
gc.collect()

if ray.is_initialized():
    print("[CLEANUP] Ray is initialized")
    # This does not kill Ray. It just helps drop local notebook references
    # if variables are deleted first.
else:
    print("[CLEANUP] Ray not initialized")

# Delete known giant variables if they exist.
for name in [
    "audio", "out", "chunks", "processed_chunks", "result_refs",
    "chunk_refs", "pending", "done", "batch"
]:
    if name in globals():
        try:
            del globals()[name]
            print(f"deleted {name}")
        except Exception as e:
            print(f"could not delete {name}: {e}")

gc.collect()
print("[DONE]")

[CLEANUP] Python garbage collection
[CLEANUP] Ray is initialized
[DONE]


In [6]:
! .\.venv\Scripts\python.exe legion_swarm_live_tools.py status

{
  "created_at": "2026-07-03T00:18:44.484235",
  "connection": {
    "ray_address": "auto",
    "namespace_requested": "legion",
    "runtime_namespace": "legion"
  },
  "registry_name": "SwarmKnowledgeRegistry",
  "registry_found": true,
  "table_count": 27,
  "tables": [
    "audio_manifest_vectors",
    "audio_vibe_gpu",
    "chris_lake_fused_raw",
    "chris_lake_omni_baseline",
    "chris_lake_raw_features",
    "chris_lake_somebody_baseline",
    "chrislake_stems_duckdb",
    "collision_results_final",
    "djsusan_stems_duckdb",
    "duckdb_agent_analysis_logs",
    "duckdb_audio_features",
    "duckdb_metadata_vectors",
    "duckdb_t_core_memory",
    "duckdb_tool_index",
    "enriched_audio_dataset",
    "enriched_samples_only",
    "enriched_tracks_only",
    "fx_sweep_test",
    "interaction_logs",
    "lancedb_audio_vibe_gpu",
    "lancedb_legion_memory",
    "legion_memory",
    "rchgen_metadata_fused",
    "skill_brain",
    "system_data_audit_report",
    "t_core_memory

2026-07-03 00:18:44,196	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 127.0.0.1:55046...
2026-07-03 00:18:44,227	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8266 


In [5]:
! .\.venv\Scripts\python.exe legion_swarm_live_tools.py important

[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\important_tables_20260703_001828.json
[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\important_tables_20260703_001828.md
 206  chris_lake_omni_baseline         rows=22 dsp=7 vector=0
 161  duckdb_audio_features            rows=1084 dsp=2 vector=0
 146  chris_lake_raw_features          rows=9 dsp=2 vector=0
 127  lancedb_audio_vibe_gpu           rows=2010 dsp=0 vector=1
 125  audio_manifest_vectors           rows=500 dsp=0 vector=3
 119  audio_vibe_gpu                   rows=1013 dsp=0 vector=1
 114  duckdb_metadata_vectors          rows=200 dsp=0 vector=2
 104  chris_lake_fused_raw             rows=65 dsp=0 vector=0
  94  duckdb_t_core_memory             rows=7618 dsp=0 vector=0
  93  enriched_audio_dataset           rows=4570 dsp=0 vector=0
  87  duckdb_tool_index                rows=3113 dsp=0 vector=0
  85  system_data_audit_report         rows=5 dsp=0 vector=0
  83  enriched_samples_only            rows=3983 dsp=0 vector=0


2026-07-03 00:18:28,589	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 127.0.0.1:55046...
2026-07-03 00:18:28,623	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8266 


In [7]:
! .\.venv\Scripts\python.exe legion_swarm_live_tools.py table duckdb_audio_features
! .\.venv\Scripts\python.exe legion_swarm_live_tools.py export duckdb_audio_features --format parquet

[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\table_duckdb_audio_features_20260703_001916.json
[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\table_duckdb_audio_features_20260703_001916.md


2026-07-03 00:19:16,228	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 127.0.0.1:55046...
2026-07-03 00:19:16,258	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8266 


[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\table_exports\duckdb_audio_features_20260703_001920.parquet
[saved] C:\WEB CASE STUDY\ray_existing_swarm_exports\table_exports\duckdb_audio_features_20260703_001920_manifest.json


2026-07-03 00:19:20,465	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 127.0.0.1:55046...
2026-07-03 00:19:20,492	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8266 


[Ray] namespace='legion'
[Registry] SwarmKnowledgeRegistry found
[Registry] live tables in summary: 27
[tables_df] ready


,table,rows,ncols,has_track_name,has_audio_filepath,has_asset_type,has_drum_type,has_vector,has_semantic_text,has_dsp_rms,has_rms_db,has_rms,has_crest,has_centroid,name_mentions_fused,name_mentions_omni,name_mentions_baseline
0,duckdb_t_core_memory,7618,8,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,enriched_audio_dataset,4570,8,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,enriched_samples_only,3983,8,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,t_core_memory,3560,6,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,duckdb_tool_index,3113,5,False,False,False,False,False,False,False,False,False,False,False,False,False,False
5,lancedb_audio_vibe_gpu,2010,6,False,False,False,True,True,False,False,False,False,False,False,False,False,False
6,duckdb_audio_features,1084,8,False,False,True,True,False,False,False,True,False,True,False,False,False,False
7,audio_vibe_gpu,1013,3,False,False,False,False,True,False,False,False,False,False,False,False,False,False
8,enriched_tracks_only,587,8,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9,audio_manifest_vectors,500,4,False,False,False,False,True,True,False,False,False,False,False,False,False,False


[classified_df] ready


,table,rows,ncols,score_full_track,score_sample_stem,score_fused,score_vector,score_baseline,score_training
0,chris_lake_fused_raw,65,8,15,0,10,0,0,0
1,rchgen_metadata_fused,60,7,0,0,5,5,0,0
2,collision_results_final,51,7,0,5,2,0,0,0
3,chris_lake_omni_baseline,22,8,0,0,0,0,12,0
4,chris_lake_somebody_baseline,22,2,0,0,0,0,5,0
5,audio_manifest_vectors,500,4,0,0,0,10,0,0
6,duckdb_metadata_vectors,200,4,0,0,0,8,0,0
7,legion_memory,2,5,0,0,0,8,0,0
8,lancedb_audio_vibe_gpu,2010,6,0,5,0,5,0,0
9,audio_vibe_gpu,1013,3,0,0,0,5,0,0



=== Candidate full-track / fused / mastering tables ===


...


=== Tables with DSP/mastering columns, regardless of source type ===


,table,rows,ncols,columns
0,duckdb_audio_features,1084,8,"[filepath, filename, asset_type, drum_type, te..."
1,chris_lake_fused_raw,65,8,"[track_name, apple_artist, apple_genre, apple_..."
2,chris_lake_omni_baseline,22,8,"[segment_name, rms, crest_factor, zero_crossin..."
3,chris_lake_raw_features,9,8,"[filepath, filename, asset_type, drum_type, te..."
4,djsusan_stems_duckdb,6,8,"[filepath, filename, asset_type, drum_type, te..."
5,chrislake_stems_duckdb,4,8,"[filepath, filename, asset_type, drum_type, te..."



=== Tables with semantic/vector columns ===


,table,rows,ncols,columns
0,lancedb_audio_vibe_gpu,2010,6,"[vector, filename, filepath, source_type, drum..."
1,audio_vibe_gpu,1013,3,"[id, filename, vector]"
2,audio_manifest_vectors,500,4,"[vector, text, metadata, type]"
3,duckdb_metadata_vectors,200,4,"[vector, text, table, type]"
4,rchgen_metadata_fused,60,7,"[file_name, file_path, functions, classes, doc..."
5,skill_brain,20,5,"[skill_name, path, essence, vector, type]"
6,legion_memory,2,5,"[id, role, text, vector, timestamp]"



Helpers ready:
  inspect_table('chris_lake_fused_raw')
  inspect_table('system_data_audit_report')
  inspect_table('duckdb_audio_features')
  search_tables(name_contains=['fused'])
  search_tables(must_have_any_cols=['track_name','audio_filepath','dsp_rms','dsp_crest'])


In [18]:
# =============================================================================
# ONE CELL: Find the real connected/mastering/Omni tables in the existing Ray swarm
# =============================================================================
# No actor creation.
# No new .py file.
# No shell commands.
# No pretending sample tables are mastering tables.
#
# This cell:
#   1. connects to existing Ray namespace "legion"
#   2. reads SwarmKnowledgeRegistry
#   3. scores tables by whether they are likely:
#        - full-track
#        - fused/joined
#        - mastering/DSP
#        - semantic/vector/Omni
#        - sample/stem/library
#   4. previews the top candidates automatically
#   5. prints one final recommendation
# =============================================================================

from __future__ import annotations

import json
import math
from typing import Any

import numpy as np
import pandas as pd
import ray


RAY_ADDRESS = "auto"
RAY_NAMESPACE = "legion"
REGISTRY_NAME = "SwarmKnowledgeRegistry"

if not ray.is_initialized():
    ray.init(address=RAY_ADDRESS, namespace=RAY_NAMESPACE, ignore_reinit_error=True)

registry = ray.get_actor(REGISTRY_NAME, namespace=RAY_NAMESPACE)
summary = ray.get(registry.get_registered_tables_summary.remote())

print(f"[Ray] namespace={ray.get_runtime_context().namespace!r}")
print(f"[Registry] {REGISTRY_NAME} found")
print(f"[Registry] live tables: {len(summary)}")


def get_table_df(name: str, max_rows: int | None = None) -> pd.DataFrame:
    obj = ray.get(registry.get_table.remote(name))

    if hasattr(obj, "slice") and hasattr(obj, "to_pandas"):
        if max_rows is not None:
            return obj.slice(0, max_rows).to_pandas()
        return obj.to_pandas()

    if isinstance(obj, pd.DataFrame):
        return obj.head(max_rows).copy() if max_rows else obj.copy()

    df = pd.DataFrame(obj)
    return df.head(max_rows).copy() if max_rows else df


def score_table(name: str, info: dict[str, Any]) -> dict[str, Any]:
    cols = list(info.get("columns", []) or [])
    colset = set(cols)
    rows = int(info.get("rows", 0) or 0)
    lname = name.lower()
    ctext = " ".join(cols).lower()

    # Intent scores.
    full_track = 0
    fused = 0
    mastering_dsp = 0
    vector_omni = 0
    sample_stem = 0
    memory_tool = 0

    # Full-track identity / metadata.
    for c in ["track_name", "audio_filepath", "apple_artist", "apple_genre", "apple_release_date"]:
        if c in colset:
            full_track += 4

    for c in ["discogs_genre", "discogs_style", "spotify_streams", "spotify_chart_position", "lb_listens", "lb_listeners", "lb_trending"]:
        if c in colset:
            full_track += 3

    # Fused/joined clues.
    if "fused" in lname:
        fused += 8
    if "collision_score" in colset:
        fused += 3
    if {"track_name", "audio_filepath"}.intersection(colset) and {"dsp_rms", "dsp_crest", "rms", "rms_db", "crest_factor"}.intersection(colset):
        fused += 8
    if {"apple_artist", "discogs_genre", "lb_listens"}.intersection(colset) and {"dsp_rms", "dsp_centroid", "spectral_centroid"}.intersection(colset):
        fused += 6

    # DSP/mastering feature clues.
    dsp_cols = [
        "dsp_rms", "dsp_crest", "dsp_sub", "dsp_bass", "dsp_mid", "dsp_high", "dsp_centroid",
        "rms", "rms_db", "crest_factor",
        "sub_bass_energy", "bass_energy", "mid_energy", "high_energy",
        "spectral_centroid", "spectral_rolloff", "spectral_bandwidth", "spectral_flatness",
        "spectral_contrast", "zero_crossing_rate", "zcr",
        "onset_strength", "transient_density", "harmonic_ratio", "percussive_ratio",
    ]
    mastering_dsp += sum(1 for c in dsp_cols if c in colset)

    if any(c.startswith("mfcc_") for c in cols) or "mfcc_vector" in colset:
        mastering_dsp += 4
    if any(c.startswith("chroma_") for c in cols) or "chroma_vector" in colset:
        mastering_dsp += 4

    # Vector / Omni / semantic clues.
    for c in ["vector", "semantic_text", "text", "metadata", "mfcc_vector", "chroma_vector"]:
        if c in colset:
            vector_omni += 4

    for token in ["omni", "semantic", "vector", "baseline", "manifest", "vibe"]:
        if token in lname:
            vector_omni += 3

    if "segment_name" in colset:
        vector_omni += 5
    if "baseline" in lname:
        vector_omni += 5

    # Sample/stem/library clues.
    if "asset_type" in colset:
        sample_stem += 4
    if "drum_type" in colset:
        sample_stem += 4
    if "filename" in colset and "filepath" in colset and "track_name" not in colset:
        sample_stem += 3
    for token in ["stem", "sample", "duckdb_audio_features", "raw_features"]:
        if token in lname:
            sample_stem += 3

    # Tool/memory noise.
    for token in ["tool", "memory", "log", "audit"]:
        if token in lname:
            memory_tool += 4

    # The whole point: connected/mastering candidates should beat raw sample/stem tables.
    connected_score = (
        full_track * 3
        + fused * 4
        + mastering_dsp * 2
        + vector_omni * 2
        - sample_stem * 2
        - memory_tool
    )

    return {
        "table": name,
        "rows": rows,
        "ncols": len(cols),
        "full_track": full_track,
        "fused": fused,
        "mastering_dsp": mastering_dsp,
        "vector_omni": vector_omni,
        "sample_stem": sample_stem,
        "memory_tool": memory_tool,
        "connected_score": connected_score,
        "columns": cols,
    }


scores = pd.DataFrame([score_table(name, info) for name, info in summary.items()])
scores = scores.sort_values("connected_score", ascending=False).reset_index(drop=True)

print("\n=== Ranked tables by likely CONNECTED mastering/Omni usefulness ===")
display(scores[[
    "table", "rows", "ncols",
    "connected_score",
    "full_track", "fused", "mastering_dsp",
    "vector_omni", "sample_stem", "memory_tool"
]])

# Pick candidates automatically.
candidate_names = scores[
    (scores["connected_score"] > 0)
    & (
        (scores["full_track"] > 0)
        | (scores["fused"] > 0)
        | (scores["vector_omni"] > 8)
    )
]["table"].head(8).tolist()

print("\n=== Auto-selected candidates to preview ===")
print(candidate_names)

candidate_previews = {}

for name in candidate_names:
    print("\n" + "=" * 100)
    print(f"TABLE: {name}")
    print("=" * 100)

    info = summary[name]
    print(f"rows={info.get('rows')} columns={len(info.get('columns', []))}")
    print("columns:")
    print(info.get("columns", []))

    try:
        df = get_table_df(name, max_rows=8)
        candidate_previews[name] = df

        # Show compact useful columns first.
        preferred = [
            "track_name", "apple_artist", "apple_genre", "audio_filepath",
            "filename", "filepath", "asset_type", "drum_type",
            "audio_tempo", "tempo", "audio_key", "key",
            "dsp_rms", "dsp_crest", "dsp_sub", "dsp_bass", "dsp_mid", "dsp_high", "dsp_centroid",
            "rms", "rms_db", "crest_factor",
            "spectral_centroid", "sub_bass_energy", "bass_energy", "mid_energy", "high_energy",
            "semantic_text", "text", "vector",
            "segment_name", "mfcc_vector", "chroma_vector",
            "collision_score", "completeness_score",
        ]
        show_cols = [c for c in preferred if c in df.columns]
        if not show_cols:
            show_cols = list(df.columns[:12])

        display(df[show_cols])

        if "asset_type" in df.columns:
            try:
                full_df = get_table_df(name)
                print("asset_type counts:")
                display(full_df["asset_type"].fillna("NULL").astype(str).value_counts().head(20).to_frame("count"))
            except Exception as e:
                print("asset_type count failed:", type(e).__name__, e)

        if "drum_type" in df.columns:
            try:
                full_df = get_table_df(name)
                print("drum_type counts:")
                display(full_df["drum_type"].fillna("NULL").astype(str).value_counts().head(20).to_frame("count"))
            except Exception as e:
                print("drum_type count failed:", type(e).__name__, e)

    except Exception as e:
        print(f"[ERROR previewing {name}] {type(e).__name__}: {e}")


# Final recommendation, based on schema only.
print("\n" + "#" * 100)
print("# RECOMMENDATION")
print("#" * 100)

top = scores.iloc[0].to_dict()
print(f"Top scored table: {top['table']}")
print(f"connected_score={top['connected_score']}")
print(f"rows={top['rows']} ncols={top['ncols']}")
print(f"full_track={top['full_track']} fused={top['fused']} mastering_dsp={top['mastering_dsp']} vector_omni={top['vector_omni']} sample_stem={top['sample_stem']}")

print("\nDo NOT build around duckdb_audio_features unless the goal is sample/stem/library physics.")
print("Build around the top fused/full-track/Omni candidate above, then use sample/stem tables only as support.")

[Ray] namespace='legion'
[Registry] SwarmKnowledgeRegistry found
[Registry] live tables: 27

=== Ranked tables by likely CONNECTED mastering/Omni usefulness ===


,table,rows,ncols,connected_score,full_track,fused,mastering_dsp,vector_omni,sample_stem,memory_tool
0,chris_lake_fused_raw,65,8,150,20,22,1,0,0,0
1,chris_lake_omni_baseline,22,8,46,0,0,7,16,0,0
2,rchgen_metadata_fused,60,7,40,0,8,0,4,0,0
3,audio_manifest_vectors,500,4,36,0,0,0,18,0,0
4,duckdb_metadata_vectors,200,4,22,0,0,0,11,0,0
5,chris_lake_somebody_baseline,22,2,16,0,0,0,8,0,0
6,audio_vibe_gpu,1013,3,14,0,0,0,7,0,0
7,legion_memory,2,5,12,0,0,0,8,0,4
8,skill_brain,20,5,8,0,0,0,4,0,0
9,enriched_audio_dataset,4570,8,0,0,0,0,0,0,0



=== Auto-selected candidates to preview ===
['chris_lake_fused_raw', 'chris_lake_omni_baseline', 'rchgen_metadata_fused', 'audio_manifest_vectors', 'duckdb_metadata_vectors']

TABLE: chris_lake_fused_raw
rows=65 columns=8
columns:
['track_name', 'apple_artist', 'apple_genre', 'apple_release_date', 'audio_filepath', 'audio_tempo', 'audio_key', 'dsp_rms']


,track_name,apple_artist,apple_genre,audio_filepath,audio_tempo,audio_key,dsp_rms,dsp_crest,dsp_sub,dsp_bass,dsp_mid,dsp_high,dsp_centroid
0,I Want You,Chris Lake,Electronic,E:\music\HOUSE\Traxsource Top 200 Tech House o...,129.199219,G#,-10.658305,3.370763,40.455685,11.039643,2.414781,1.254643,3281.762207
1,Ease My Mind,Chris Lake & Abel Balder,House,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,F#,-8.626541,2.828623,49.493317,29.684353,4.019258,1.493119,2570.350586
2,Toxic,Chris Lake & Ragie Ban,Dance,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,F,-11.412646,4.006662,36.485493,17.768583,2.885926,1.143775,2689.214844
3,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,E:\music\HOUSE\Beatport Tech House Top 100 Dec...,129.199219,D,-14.317103,5.638170,20.563969,15.175913,1.552364,0.656931,2331.311523
4,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,E:\music\HOUSE\Beatport Top 100 Deep House Jan...,129.199219,F#,-11.225826,3.589428,40.509830,14.715528,1.753373,0.636549,2084.519531
5,Somebody (feat. Kimbra & Sante Sansone),"Gotye, FISHER & Chris Lake",Dance,E:\music\HOUSE\Beatport Top 100 Tech House Nov...,129.199219,F#,-11.834819,4.113777,27.087049,18.817310,1.263495,0.830222,2501.069824
6,Summertime Blues,"Chris Lake, Sammy Virji & Nathan Nicholson",Dance,E:\music\HOUSE\Beatport Top 100 Tech House Nov...,129.199219,A,-11.011549,3.704059,25.308617,15.438262,3.453703,1.850543,3317.910156
7,Psycho,Chris Lake,Dance,E:\music\HOUSE\Beatport Top 100 Downloads Nove...,129.199219,G#,-11.290215,4.243157,34.448639,16.418251,2.700692,1.721593,3304.476074



TABLE: chris_lake_omni_baseline
rows=22 columns=8
columns:
['segment_name', 'rms', 'crest_factor', 'zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness']


...


TABLE: audio_manifest_vectors
rows=500 columns=4
columns:
['vector', 'text', 'metadata', 'type']


,text,vector
0,Audio: Claps & Snaps(03) - A#m - 128-1.wav. P...,[Vector dimension: 768]
1,Audio: Claps & Snaps(03) - A#m - 128.wav. Pat...,[Vector dimension: 768]
2,Audio: $IS030GL.mp3. Path: C:\Users\adams\Musi...,[Vector dimension: 768]
3,Audio: ._ Claps & Snaps(03) - A#m - 128-1.wav....,[Vector dimension: 768]
4,Audio: ._ Claps & Snaps(03) - A#m - 128.wav. P...,[Vector dimension: 768]
5,Audio: ._003 Snare - Zenhiser PHD.wav. Path: C...,[Vector dimension: 768]
6,Audio: ._017 Hi Hat - Zenhiser PHD.wav. Path: ...,[Vector dimension: 768]
7,Audio: ._12-Audio 0001 [2025-10-03 034247]-1.w...,[Vector dimension: 768]



TABLE: duckdb_metadata_vectors
rows=200 columns=4
columns:
['vector', 'text', 'table', 'type']


,text,vector
0,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
1,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
2,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
3,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
4,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
5,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
6,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]
7,Source: DuckDB. Table: t_core_memory. Data: fi...,[Vector dimension: 768]



####################################################################################################
# RECOMMENDATION
####################################################################################################
Top scored table: chris_lake_fused_raw
connected_score=150
rows=65 ncols=8
full_track=20 fused=22 mastering_dsp=1 vector_omni=0 sample_stem=0

Do NOT build around duckdb_audio_features unless the goal is sample/stem/library physics.
Build around the top fused/full-track/Omni candidate above, then use sample/stem tables only as support.


In [2]:

#Initializing the Generative Omni-Code Genome (PyTorch)...
CodeGenomeAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=770, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=256, out_features=770, bias=True)
  )
)

# OMNI-GENOMIC INFERENCE TEST:
Mutated/Synthetic Footprint dims: torch.Size([1, 770])

#🔮 The Generative Model just hallucinated a file with:
   Generated Size: 2067.72 KB
   Generated Rows: 2
   Generated Semantic Meaning: [768-D Embedding Tensor hallucinated successfully]

SyntaxError: invalid syntax (3401028479.py, line 3)